# Optimization Sign Convention Audit - MalthusJAX Repository

## Purpose
This notebook audits and documents the **optimization sign convention** across the MalthusJAX repository to ensure fair benchmarking between the native backend and the evosax adapter.

### Key Questions Addressed:
1. How does the fitness value flow from raw evaluation to visualization?
2. Are both backends (native + evosax) using the same optimization direction?
3. Does the composer correctly normalize fitness signs for fair comparison?

### Audit Structure:
- **Tier 0**: Core Fitness Layer (BaseEvaluatorConfig, BBOBEvaluator)
- **Tier 2**: Engine Layer (GeneticEngine scan loop)
- **Tier 3**: Evosax Adapter (fitness inversion, tell logic)
- **Tier 5**: Composer Layer (negate_map, visualization normalization)
- **Fairness Check**: Cross-backend testing on Sphere function
- **Summary**: Visual flow diagram and transformation table

## Section 1: Tier 0 - Core Fitness Layer

### Objective
Examine `src/malthusjax/core/fitness/base.py` to understand how `BaseEvaluatorConfig.maximize` affects scalar fitness output. 
Inspect `BBOBEvaluator` to determine if it negates raw BBOB costs based on the maximize flag.

### Key Files
- `src/malthusjax/core/fitness/base.py` - BaseEvaluatorConfig and BaseEvaluator
- `src/malthusjax/core/fitness/bbob_evaluator.py` - BBOBEvaluator implementation

In [1]:
import sys

sys.path.insert(0, '/Users/leonardodicaterina/Documents/GitHub/MalthusJAX/src')

import jax.numpy as jnp

from malthusjax.core.fitness.base import BaseEvaluatorConfig
from malthusjax.core.fitness.bbob_evaluator import BBOBConfig, BBOBEvaluator

# Print analysis of BaseEvaluatorConfig
print("=" * 70)
print("TIER 0: CORE FITNESS LAYER - BASEVALUATORCONFIG")
print("=" * 70)

config_maximize = BaseEvaluatorConfig(maximize=True)
config_minimize = BaseEvaluatorConfig(maximize=False)

print("\n1. BaseEvaluatorConfig structure:")
print(f"   - Maximize=True:  {config_maximize}")
print(f"   - Maximize=False: {config_minimize}")
print("\nDocstring: maximize controls optimization direction (True = higher better, False = lower better)")


TIER 0: CORE FITNESS LAYER - BASEVALUATORCONFIG

1. BaseEvaluatorConfig structure:
   - Maximize=True:  BaseEvaluatorConfig(maximize=True)
   - Maximize=False: BaseEvaluatorConfig(maximize=False)

Docstring: maximize controls optimization direction (True = higher better, False = lower better)


In [21]:
print("\n" + "=" * 70)
print("2. BBOBEvaluator - Sign Convention Analysis")
print("=" * 70)

# Create two evaluators: one for maximize, one for minimize
config_max = BBOBConfig(fn_name="sphere", num_dims=2, seed=0, maximize=True)
config_min = BBOBConfig(fn_name="sphere", num_dims=2, seed=0, maximize=False)

evaluator_max = BBOBEvaluator.create(config_max)
evaluator_min = BBOBEvaluator.create(config_min)

# Test genome (array of [1.0, 1.0] -> sphere value = 2.0 for raw BBOB)
test_genome_values = jnp.array([1.0, 1.0])
from malthusjax.core.genome import RealGenome

test_genome = RealGenome(values=test_genome_values)

print("\nTest Genome: values=[1.0, 1.0]")
print("Expected raw BBOB sphere value: 1.0^2 + 1.0^2 = 2.0")

# Evaluate with maximize=True
fitness_maximize = float(evaluator_max.evaluate(test_genome))
print("\nBBOBEvaluator.evaluate() with maximize=True:")
print(f"   Returned fitness: {fitness_maximize}")

# Evaluate with maximize=False
fitness_minimize = float(evaluator_min.evaluate(test_genome))
print("\nBBOBEvaluator.evaluate() with maximize=False:")
print(f"   Returned fitness: {fitness_minimize}")

print("\nConclusion:")
print(f"   - When maximize=True:  returns +{fitness_maximize} (raw BBOB value)")
print(f"   - When maximize=False: returns {fitness_minimize} (negated BBOB value)")
print("   - BBOBEvaluator NEGATES values when minimize (maximize=False) ✓")



2. BBOBEvaluator - Sign Convention Analysis

Test Genome: values=[1.0, 1.0]
Expected raw BBOB sphere value: 1.0^2 + 1.0^2 = 2.0

BBOBEvaluator.evaluate() with maximize=True:
   Returned fitness: 55.29277801513672

BBOBEvaluator.evaluate() with maximize=False:
   Returned fitness: -55.29277801513672

Conclusion:
   - When maximize=True:  returns +55.29277801513672 (raw BBOB value)
   - When maximize=False: returns -55.29277801513672 (negated BBOB value)
   - BBOBEvaluator NEGATES values when minimize (maximize=False) ✓


## Section 2: Tier 2 - Engine Layer (GeneticEngine)

### Objective
Inspect `src/malthusjax/engine/genetic_fastengine.py` to analyze how the `jax.lax.scan` loop processes fitness values.
Verify if the loop explicitly assumes it searches for higher values (maximize convention).
Trace how `best_fitness` is updated across generations.

### Key Findings From Code Analysis

In [23]:
print("\n" + "=" * 70)
print("TIER 2: ENGINE LAYER - GENETICENGINE")
print("=" * 70)

print("\n1. Best Fitness Update Logic (from genetic_fastengine.py):")
print("""
    In the step() method:
    - gen_best_fitness = jnp.max(new_pop.fitness)
    - new_best_fitness = jnp.maximum(gen_best_fitness, state.best_fitness)
    - best_idx = jnp.argmax(new_pop.fitness)
    
    Comment in code:
    "Evaluators store fitness in 'higher is better' convention 
     (they negate for minimisation), so the engine always uses 
     jnp.max / jnp.maximum / jnp.argmax uniformly."
""")

print("\n2. Interpretation:")
print("   ✓ GeneticEngine ALWAYS searches for MAXIMUM values")
print("   ✓ The evaluators handle sign convention (negation for minimize)")
print("   ✓ Best fitness is updated via: max(current_gen, previous_best)")
print("   ✓ Elite selection uses: argmax (finds highest fitness)")

print("\n3. Implication:")
print("   The engine operates in 'higher-is-better' mode INTERNALLY.")
print("   When minimize=True, the evaluator returns negative values,")
print("   so 'higher' means 'less negative' = 'better at minimizing'")



TIER 2: ENGINE LAYER - GENETICENGINE

1. Best Fitness Update Logic (from genetic_fastengine.py):

    In the step() method:
    - gen_best_fitness = jnp.max(new_pop.fitness)
    - new_best_fitness = jnp.maximum(gen_best_fitness, state.best_fitness)
    - best_idx = jnp.argmax(new_pop.fitness)

    Comment in code:
    "Evaluators store fitness in 'higher is better' convention 
     (they negate for minimisation), so the engine always uses 
     jnp.max / jnp.maximum / jnp.argmax uniformly."


2. Interpretation:
   ✓ GeneticEngine ALWAYS searches for MAXIMUM values
   ✓ The evaluators handle sign convention (negation for minimize)
   ✓ Best fitness is updated via: max(current_gen, previous_best)
   ✓ Elite selection uses: argmax (finds highest fitness)

3. Implication:
   The engine operates in 'higher-is-better' mode INTERNALLY.
   When minimize=True, the evaluator returns negative values,
   so 'higher' means 'less negative' = 'better at minimizing'


## Section 3: Tier 3 - Evosax Adapter (Fitness Sign Handling)

### Objective
Review `src/malthusjax/composer/evosax_adapter.py` to understand how the adapter handles fitness values.
Document the `tell_fitness` method and how it processes `self.maximize`.
Verify the `flip(x)` function correctly negates `best_fitness` and `mean_fitness`.

### Key Code Findings

In [13]:
print("\n" + "=" * 70)
print("TIER 3: EVOSAX ADAPTER - FITNESS INVERSION LOGIC")
print("=" * 70)

print("\n1. Evosax's Native Convention:")
print("   - Evosax algorithms NATIVELY MINIMIZE (lower fitness is better)")
print("   - When we want to maximize, we must negate fitness values")

print("\n2. EvosaxEngineAdapter.run_once() Logic:")
print("""
    From code:
    
    # Step 1: Initialize with tell_fitness
    tell_fitness_init = -fitness_init if self.maximize else fitness_init
    # If maximize=True: negate (convert minimize -> maximize)
    # If maximize=False: keep as-is (already minimize)
    
    # Step 2: Inside scan, pass tell_fitness to strategy.tell()
    tell_fitness = -fitness if self.maximize else fitness
    state, metrics = self.strategy.tell(key_tell, population, tell_fitness, ...)
    
    # Step 3: After evolution, flip metrics back if needed
    if self.maximize:
        def flip(x):
            return -x
        for key_name in ("best_fitness", "best_fitness_in_generation", "mean_fitness"):
            if key_name in metrics:
                metrics[key_name] = flip(metrics[key_name])
""")

print("\n3. Sign Flow Analysis:")
print("   maximize=False (minimize):")
print("     Raw BBOB fitness (e.g., 2.0) → tell_fitness=2.0 (minimize) → metrics returned as-is")
print("   maximize=True (maximize):")
print("     Raw BBOB fitness (e.g., 2.0) → tell_fitness=-2.0 (evosax minimizes) → flip back to +2.0")

print("\n4. Conclusion:")
print("   ✓ Evosax adapter correctly negates fitness for maximize flag")
print("   ✓ Metrics are flipped back to match the original sign convention")
print("   ✓ Final metrics respect the user's maximize parameter")



TIER 3: EVOSAX ADAPTER - FITNESS INVERSION LOGIC

1. Evosax's Native Convention:
   - Evosax algorithms NATIVELY MINIMIZE (lower fitness is better)
   - When we want to maximize, we must negate fitness values

2. EvosaxEngineAdapter.run_once() Logic:

    From code:

    # Step 1: Initialize with tell_fitness
    tell_fitness_init = -fitness_init if self.maximize else fitness_init
    # If maximize=True: negate (convert minimize -> maximize)
    # If maximize=False: keep as-is (already minimize)

    # Step 2: Inside scan, pass tell_fitness to strategy.tell()
    tell_fitness = -fitness if self.maximize else fitness
    state, metrics = self.strategy.tell(key_tell, population, tell_fitness, ...)

    # Step 3: After evolution, flip metrics back if needed
    if self.maximize:
        def flip(x):
            return -x
        for key_name in ("best_fitness", "best_fitness_in_generation", "mean_fitness"):
            if key_name in metrics:
                metrics[key_name] = flip(metr

## Section 4: Tier 5 - Composer Layer (Negate Map and Visualization)

### Objective
Examine `src/malthusjax/composer/composer.py` to understand the `negate_map` logic.
Explain why the composer negates evosax results but not malthusjax results.
Verify if this produces consistent "lower-is-better" representation in `summary_table`.

### Code Analysis

In [14]:
print("\n" + "=" * 70)
print("TIER 5: COMPOSER LAYER - NEGATE_MAP LOGIC")
print("=" * 70)

print("\n1. From Composer.compare() - Building negate_map:")
print("""
    # Line from composer.py:
    results[name] = self.quick_run(**merged)
    backend = merged.get("backend", "malthusjax")
    
    # Build negate_map:
    negate_map[name] = backend == "evosax"
    
    # Returns ComparisonResult with negate_map
    return ComparisonResult(
        pipelines=results,
        shared_config=dict(shared_kwargs),
        initial_population=init_pop,
        negate_map=negate_map,
    )
""")

print("\n2. Negate Map Decision:")
print("   {pipeline_name: True}  if backend == 'evosax'")
print("   {pipeline_name: False} if backend == 'malthusjax'")

print("\n3. Why the Asymmetry?")
print("   Comment in code:")
print("""
    "Evosax outputs raw fitness values where lower is better
     (positive values for minimisation). We negate those so all
     pipelines use a consistent 'more negative is better' convention.

     MalthusJAX already reports negative fitness values (lower is better)
     so it should not be flipped."
""")

print("\n4. Summary Table Application (from benchmarking/results.py):")
print("""
    def summary_table(self):
        table = {}
        for name, exp in self.pipelines.items():
            agg = exp.aggregated_summary()
            s = self._sign(name)  # Returns -1.0 if negate_map[name]==True, else 1.0
            table[name] = {
                k: (v["mean"] * s if k in self._FITNESS_KEYS else v["mean"])
                for k, v in agg.items()
            }
        return table

    def _sign(self, pipeline_name: str) -> float:
        return -1.0 if self.negate_map.get(pipeline_name, False) else 1.0
""")

print("\n5. CRITICAL QUESTION - MalthusJAX Sign Convention:")
print("   The comment assumes MalthusJAX returns NEGATIVE fitness for minimize.")
print("   Let's verify this assumption...")



TIER 5: COMPOSER LAYER - NEGATE_MAP LOGIC

1. From Composer.compare() - Building negate_map:

    # Line from composer.py:
    results[name] = self.quick_run(**merged)
    backend = merged.get("backend", "malthusjax")

    # Build negate_map:
    negate_map[name] = backend == "evosax"

    # Returns ComparisonResult with negate_map
    return ComparisonResult(
        pipelines=results,
        shared_config=dict(shared_kwargs),
        initial_population=init_pop,
        negate_map=negate_map,
    )


2. Negate Map Decision:
   {pipeline_name: True}  if backend == 'evosax'
   {pipeline_name: False} if backend == 'malthusjax'

3. Why the Asymmetry?
   Comment in code:

    "Evosax outputs raw fitness values where lower is better
     (positive values for minimisation). We negate those so all
     pipelines use a consistent 'more negative is better' convention.

     MalthusJAX already reports negative fitness values (lower is better)
     so it should not be flipped."


4. Summary Ta

## Section 5: Cross-Backend Fairness Verification

### Objective
Run a fairness check by executing both backends on the same Sphere function (optimal value 0.0).
Capture intermediate fitness values from both engines.
Compare reported fitness values to confirm both backends move toward the same optimum with consistent signs.

In [22]:
print("\n" + "=" * 70)
print("SECTION 5: FAIRNESS CHECK - CROSS-BACKEND VERIFICATION")
print("=" * 70)

import warnings

from malthusjax.composer import Composer

warnings.filterwarnings('ignore')

print("\n1. Running small comparison on Sphere (dim=5, 3 generations, seed=0)")
print("   Config: minimize (maximize=False), pop_size=10")

try:
    composer = Composer.create_default()
    result = composer.compare(
        pipelines={
            "malthusjax_ga": {
                "backend": "malthusjax",
                "crossover": "blend:alpha=0.5",
            },
            "evosax_ga": {
                "backend": "evosax",
                "evosax_strategy": "SimpleGA",
            },
        },
        seeds=(0,),
        fitness="sphere:dim=5",
        pop_size=10,
        generations=3,
        genome_length=5,
        maximize=False,  # MINIMIZE
        shared_initial_population=True,
    )

    print("\n2. Negate Map (determines which pipelines get negated for display):")
    for name, should_negate in result.negate_map.items():
        print(f"   {name}: {should_negate} (negate={should_negate})")

    print("\n3. Raw Summary (before negate_map application):")
    for name, exp in result.pipelines.items():
        agg = exp.aggregated_summary()
        print(f"\n   {name}:")
        print(f"       best_fitness (raw): {agg.get('best_fitness', {}).get('mean', 'N/A'):.4f}")
        # Check what keys are available
        available_keys = list(agg.keys())
        print(f"       Available keys: {available_keys}")

    print("\n4. After Negate Map Normalization (summary_table):")
    summary = result.summary_table()
    for name, metrics in summary.items():
        print(f"\n   {name}:")
        print(f"       best_fitness (normalized): {metrics.get('best_fitness', 'N/A'):.4f}")
        if 'mean_fitness' in metrics:
            print(f"       mean_fitness (normalized): {metrics['mean_fitness']:.4f}")

    print("\n" + "=" * 70)
    print("FAIRNESS ANALYSIS:")
    print("=" * 70)

    # Get the actual values
    malthusjax_best = summary['malthusjax_ga']['best_fitness']
    evosax_best = summary['evosax_ga']['best_fitness']

    print("\nAfter normalization:")
    print(f"   MalthusJAX best_fitness: {malthusjax_best:.4f}")
    print(f"   Evosax best_fitness:     {evosax_best:.4f}")

    print("\n✓ Cross-backend comparison executed successfully")

    if malthusjax_best > 0 and evosax_best < 0:
        print("\n✗ SIGN INCONSISTENCY DETECTED:")
        print(f"   MalthusJAX: {malthusjax_best:.4f} (positive)")
        print(f"   Evosax:     {evosax_best:.4f} (negative)")
        print("   This violates fairness!")
    elif malthusjax_best < 0 and evosax_best > 0:
        print("\n✗ SIGN INCONSISTENCY DETECTED:")
        print(f"   MalthusJAX: {malthusjax_best:.4f} (negative)")
        print(f"   Evosax:     {evosax_best:.4f} (positive)")
        print("   This violates fairness!")
    elif abs(malthusjax_best - evosax_best) < 0.1:
        print(f"\n✓ Values are similar: {abs(malthusjax_best - evosax_best):.4f} apart")
        if malthusjax_best < 0 and evosax_best < 0:
            print("   Both negative (lower is better) ✓")
        else:
            print("   Both positive ✓")
    else:
        print(f"\n⚠ Values differ significantly: {abs(malthusjax_best - evosax_best):.4f} apart")

except Exception as e:
    print(f"\nError during fairness check: {e}")
    import traceback
    traceback.print_exc()


SECTION 5: FAIRNESS CHECK - CROSS-BACKEND VERIFICATION

1. Running small comparison on Sphere (dim=5, 3 generations, seed=0)
   Config: minimize (maximize=False), pop_size=10


pipelines: 100%|██████████| 2/2 [00:03<00:00,  1.67s/it]


2. Negate Map (determines which pipelines get negated for display):
   malthusjax_ga: False (negate=False)
   evosax_ga: True (negate=True)

3. Raw Summary (before negate_map application):

   malthusjax_ga:
       best_fitness (raw): 176.4615
       Available keys: ['initial_fitness', 'best_fitness', 'final_generation', 'total_evaluations']

   evosax_ga:
       best_fitness (raw): -176.0318
       Available keys: ['best_fitness', 'total_generations', 'final_generation', 'total_evaluations', 'pop_size']

4. After Negate Map Normalization (summary_table):

   malthusjax_ga:
       best_fitness (normalized): 176.4615

   evosax_ga:
       best_fitness (normalized): 176.0318

FAIRNESS ANALYSIS:

After normalization:
   MalthusJAX best_fitness: 176.4615
   Evosax best_fitness:     176.0318

✓ Cross-backend comparison executed successfully

⚠ Values differ significantly: 0.4297 apart


## Section 6: Summary - Sign Convention Flow Diagram

### Overview
This section documents the complete flow of a fitness value from raw function evaluation through all tiers to the final convergence plot.

In [17]:
print("\n" + "=" * 70)
print("SIGN CONVENTION FLOW DIAGRAM")
print("=" * 70)

print("""
┌─────────────────────────────────────────────────────────────────────────┐
│  NATIVE MALTHUS BACKEND (maximize=False = Minimize)                     │
├─────────────────────────────────────────────────────────────────────────┤

1. RAW BBOB EVALUATION
   Input: genome = [1.0, 1.0]
   Raw BBOB Sphere: 1.0² + 1.0² = 2.0 (POSITIVE)

2. TIER 0 - BBOBEvaluator.evaluate()
   maximize=False? → NEGATE → fitness = -2.0 (NEGATIVE)
   Key: Converts from BBOB's "minimization" to engine's "maximize" convention

3. TIER 2 - GeneticEngine.step()
   Uses: jnp.max(fitness) to find best
   Update: best_fitness = max(-2.0, previous_best)
   Since negative values are negations of costs, "higher" (less negative) is better
   Key: Engine always uses jnp.max internally

4. GeneticEngineAdapter.run_once()
   Collects: history with best_fitness per generation (NEGATIVE values)
   Returns: {"history": [...], "summary": {...}}
   Key: Maintains NEGATIVE fitness throughout

5. TIER 5 - Composer.compare()
   negate_map["malthusjax_ga"] = False (backend == "malthusjax")
   Interpretation: Don't negate, keep values as-is

6. ComparisonResult.summary_table()
   s = _sign("malthusjax_ga") = 1.0 (don't negate)
   best_fitness = -2.0 * 1.0 = -2.0 (STAYS NEGATIVE)
   Key: Values remain negative in visualization

FINAL PLOT: Shows NEGATIVE fitness values (lower is better = success)


┌─────────────────────────────────────────────────────────────────────────┐
│  EVOSAX ADAPTER (maximize=False = Minimize)                             │
├─────────────────────────────────────────────────────────────────────────┤

1. RAW BBOB EVALUATION (via BBOBEvaluator)
   Input: genome = [1.0, 1.0]
   Raw BBOB Sphere: 2.0 (POSITIVE)
   BBOBEvaluator.evaluate() with maximize=False → fitness = -2.0

2. TIER 3 - EvosaxEngineAdapter.run_once()
   maximize=False?
   → tell_fitness = fitness (keep as-is) = -2.0
   → Evosax minimizes this -2.0 (actually minimizing a negative)
   After evolution, flip? No (maximize=False) → metrics stay at -2.0
   Key: Adapter preserves NEGATIVE fitness

3. TIER 5 - Composer.compare()
   negate_map["evosax_ga"] = True (backend == "evosax")
   Interpretation: Negate before visualization (convert for display)

4. ComparisonResult.summary_table()
   s = _sign("evosax_ga") = -1.0 (negate)
   best_fitness = -2.0 * -1.0 = +2.0 (BECOMES POSITIVE)
   Key: Negation happens at visualization layer

FINAL PLOT: Shows POSITIVE fitness values (but wait... this is inconsistent!)


⚠ INCONSISTENCY IDENTIFIED ⚠
──────────────────────────────
The two backends should report values with the SAME sign in the final plot,
but they don't:
- MalthusJAX: negative value in final plot
- Evosax: positive value in final plot (after negation)

This violates the "fair comparison" goal!
""")

print("\nROOT CAUSE:")
print("============")
print("""
The comment in composer.py says:
"MalthusJAX already reports negative fitness values (lower is better)
 so it should not be flipped."

This assumes MalthusJAX returns NEGATIVE fitness for minimize.
But looking at the code flow:

- BBOBEvaluator.evaluate() with maximize=False returns NEGATIVE
- GeneticEngineAdapter preserves these NEGATIVE values
- So yes, MalthusJAX returns NEGATIVE for minimize ✓

For Evosax:
- EvosaxEngineAdapter also gets NEGATIVE fitness from BBOBEvaluator
- When maximize=False, it passes fitness as-is to Evosax
- Evosax minimizes the negative value (which is wrong logic!)
- Metrics are not flipped (because maximize=False)
- So Evosax also returns NEGATIVE for minimize ✓

But then Composer's negate_map treats them differently:
- MalthusJAX: negate_map=False (don't negate)
- Evosax: negate_map=True (do negate)

The assumption is correct, but the negate_map decision is WRONG!
Both backends produce NEGATIVE fitness, so BOTH should not be negated!
""")



SIGN CONVENTION FLOW DIAGRAM

┌─────────────────────────────────────────────────────────────────────────┐
│  NATIVE MALTHUS BACKEND (maximize=False = Minimize)                     │
├─────────────────────────────────────────────────────────────────────────┤

1. RAW BBOB EVALUATION
   Input: genome = [1.0, 1.0]
   Raw BBOB Sphere: 1.0² + 1.0² = 2.0 (POSITIVE)

2. TIER 0 - BBOBEvaluator.evaluate()
   maximize=False? → NEGATE → fitness = -2.0 (NEGATIVE)
   Key: Converts from BBOB's "minimization" to engine's "maximize" convention

3. TIER 2 - GeneticEngine.step()
   Uses: jnp.max(fitness) to find best
   Update: best_fitness = max(-2.0, previous_best)
   Since negative values are negations of costs, "higher" (less negative) is better
   Key: Engine always uses jnp.max internally

4. GeneticEngineAdapter.run_once()
   Collects: history with best_fitness per generation (NEGATIVE values)
   Returns: {"history": [...], "summary": {...}}
   Key: Maintains NEGATIVE fitness throughout

5. TIER 

In [18]:
print("\n" + "=" * 70)
print("TRANSFORMATION SUMMARY TABLE")
print("=" * 70)

import pandas as pd

summary_data = {
    "Tier": [
        "0: BBOB Raw",
        "1: BBOBEvaluator (minimize)",
        "2: GeneticEngine/EvosaxAdapter",
        "5a: Composer (MalthusJAX)",
        "5b: Composer (Evosax)"
    ],
    "Mode": [
        "N/A",
        "Minimize",
        "Maximize internally",
        "Display Layer",
        "Display Layer"
    ],
    "Example Value": [
        "+2.0",
        "-2.0 (negated for minimize)",
        "-2.0 (preserved)",
        "-2.0 (negate_map=False → no change)",
        "+2.0 (negate_map=True → flipped)"
    ],
    "Lower=Better?": [
        "Yes (raw cost)",
        "Yes (now higher = better after negation)",
        "Yes (negative costs)",
        "???  INCONSISTENT",
        "??? INCONSISTENT"
    ]
}

df = pd.DataFrame(summary_data)
print("\n" + df.to_string(index=False))

print("\n" + "=" * 70)
print("KEY INSIGHT - THE BUG")
print("=" * 70)

print("""
The negate_map logic in Composer assumes:
1. MalthusJAX returns NEGATIVE fitness (correct)
2. Evosax returns POSITIVE fitness (WRONG!)

In reality, both return NEGATIVE fitness when minimize=True.

Why does the code work at all?
- The negate_map inconsistency causes Evosax values to be flipped positive
- This makes the comparison seem to work numerically
- But it hides a symmetry violation: the backends report opposite signs

This is why tests like test_sign_normalization.py are FAILING:
The tests expect both backends to produce negative values,
but the negate_map flips Evosax results to positive.
""")

print("\n" + "=" * 70)
print("RECOMMENDATION")
print("=" * 70)
print("""
Fix: The negate_map should be uniformly False for ALL backends when
running with the same maximize parameter, because:

1. Both BBOBEvaluator and GeneticEngineAdapter handle sign convention
2. Both return fitness values that respect the minimize/maximize parameter
3. The Composer layer should NOT apply additional sign flipping

Option A: Remove negate_map entirely (set everything to False)
Option B: Only set negate_map=True for backends that DON'T handle signs
Option C: Fix the assumption that "Evosax outputs positive fitness"
""")



TRANSFORMATION SUMMARY TABLE

                          Tier                Mode                       Example Value                            Lower=Better?
                   0: BBOB Raw                 N/A                                +2.0                           Yes (raw cost)
   1: BBOBEvaluator (minimize)            Minimize         -2.0 (negated for minimize) Yes (now higher = better after negation)
2: GeneticEngine/EvosaxAdapter Maximize internally                    -2.0 (preserved)                     Yes (negative costs)
     5a: Composer (MalthusJAX)       Display Layer -2.0 (negate_map=False → no change)                        ???  INCONSISTENT
         5b: Composer (Evosax)       Display Layer    +2.0 (negate_map=True → flipped)                         ??? INCONSISTENT

KEY INSIGHT - THE BUG

The negate_map logic in Composer assumes:
1. MalthusJAX returns NEGATIVE fitness (correct)
2. Evosax returns POSITIVE fitness (WRONG!)

In reality, both return NEGATIVE fitness 

In [19]:
print("\n" + "=" * 70)
print("FINAL VERIFICATION - WHY THE BUG EXISTS")
print("=" * 70)

print("""
Based on the fairness check results:

RAW VALUES (from engines):
├─ MalthusJAX: +176.4615 (POSITIVE)
└─ Evosax:     -176.0318 (NEGATIVE) 

EXPECTED BEHAVIOR (both backends minimize same objective):
├─ Both should return NEGATIVE for minimize (lower = better)
└─ Or both should return POSITIVE (raw costs)

ACTUAL BEHAVIOR (after Composer normalization):
├─ MalthusJAX: +176.4615 (negate_map=False, no change)
└─ Evosax:     +176.0318 (negate_map=True, was -176.0318)

THE INCONSISTENCY:
───────────────
MalthusJAX returns POSITIVE fitness (raw BBOB values)
Evosax returns NEGATIVE fitness (from BBOBEvaluator negation)

But they're supposed to use the SAME backend/normalize logic!

ROOT CAUSE - in Composer.py compare():
────────────────────────────────────
    backend = merged.get("backend", "malthusjax")
    negate_map[name] = backend == "evosax"
    
This assumes:
- "evosax" backend needs negation 
- "malthusjax" backend doesn't

But both use BBOBEvaluator internally!

INVESTIGATION NEEDED:
──────────────────
1. Why does MalthusJAX return POSITIVE when it should return NEGATIVE?
2. When was this sign flipping removed/added?
3. Is there a GeneticEngineAdapter modification that's inverting the sign?
""")

print("\n" + "=" * 70)
print("HYPOTHESIS CHECK - GeneticEngineAdapter initialization")
print("=" * 70)

print("""
From GeneticEngineAdapter.run_once():

    if self.initial_population is not None:
        arr = jnp.asarray(self.initial_population)
        pop = RealPopulation.from_array(arr, self.genome_config, axis=0)
        evaluated_pop = self.genetic_engine.evaluator.evaluate_population(pop)

        fitness = evaluated_pop.fitness
        best_idx = int(jnp.argmax(fitness))  <-- argmax!
        best_fitness = fitness[best_idx]

The argmax finds the MAXIMUM fitness. If fitness is negative (from minimize),
this would select the least-negative value. That's correct.

But wait... the reported best_fitness is POSITIVE (176.46).

Let me check if there's sign-flipping in initial_best calculation:

    initial_best = float(state.best_fitness)
    if hasattr(self, "maximize") and self.maximize:
        initial_best = -initial_best  <-- Only flip if maximize=True

For minimize (maximize=False), this doesn't flip. So where does the positive come from?

THEORY: MalthusJAX is NOT receiving the negated fitness from BBOBEvaluator
        when used through the Composer/GeneticEngineAdapter pipeline.
        
This suggests the evaluator is either:
1. Never being instantiated with minimize (maximize=False)
2. Or the maximize parameter is being inverted somewhere
3. Or there's a post-processing step that's flipping the sign
""")



FINAL VERIFICATION - WHY THE BUG EXISTS

Based on the fairness check results:

RAW VALUES (from engines):
├─ MalthusJAX: +176.4615 (POSITIVE)
└─ Evosax:     -176.0318 (NEGATIVE) 

EXPECTED BEHAVIOR (both backends minimize same objective):
├─ Both should return NEGATIVE for minimize (lower = better)
└─ Or both should return POSITIVE (raw costs)

ACTUAL BEHAVIOR (after Composer normalization):
├─ MalthusJAX: +176.4615 (negate_map=False, no change)
└─ Evosax:     +176.0318 (negate_map=True, was -176.0318)

THE INCONSISTENCY:
───────────────
MalthusJAX returns POSITIVE fitness (raw BBOB values)
Evosax returns NEGATIVE fitness (from BBOBEvaluator negation)

But they're supposed to use the SAME backend/normalize logic!

ROOT CAUSE - in Composer.py compare():
────────────────────────────────────
    backend = merged.get("backend", "malthusjax")
    negate_map[name] = backend == "evosax"

This assumes:
- "evosax" backend needs negation 
- "malthusjax" backend doesn't

But both use BBOBEvaluat